# Example: BBBC021 — Compound Morphological Profiling

This notebook demonstrates morphological profiling of a compound screen with Mechanism of Action (MoA) annotations.

**Dataset Information:**
- **Source:** [Broad Institute BBBC021](https://bbbc.broadinstitute.org/BBBC021)
- **Description:** Multi-week fluorescent plates with MoA annotations.
- **Metadata:** Provided in `metadata.csv` and `BBBC021_MoA_table_formated.csv`.

**Workflow Overview:**
1. Load and merge metadata with MoA annotations.
2. Link 3-channel images (Actin, Tubulin, DAPI) with metadata.
3. Extract DINOv2 embeddings and compute morphological properties.
4. Analyze mechanism of action via clustering and visualization.
5. Interactively explore the 3-channel phenospace.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model
import pandas as pd
import os
from pathlib import Path

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

## 1. Paths & Configuration

Specify the paths to your BBBC021 data and metadata files.

In [ ]:
BASE_DIR = Path("data/BBBC021")
DATA_DIR = BASE_DIR / "Week123"
METADATA_PATH = BASE_DIR / "metadata.csv"
MOA_PATH = BASE_DIR / "BBBC021_MoA_table_formated.csv"

CHECKPOINT_EMBEDDINGS = "checkpoints/bbbc021_embeddings.h5"
CHECKPOINT_PROPERTIES = "checkpoints/bbbc021_properties.h5"

os.makedirs("checkpoints", exist_ok=True)

## 2. Load & Merge Metadata

We load the image metadata and merge it with the Mechanism of Action (MoA) annotations.

In [ ]:
df_metadata = pd.read_csv(METADATA_PATH)
df_moa = pd.read_csv(MOA_PATH)

# Merge to associate images with MoA
df_merged = df_metadata.merge(
    df_moa,
    on=["Image_Metadata_Compound", "Image_Metadata_Concentration"],
    how="inner",
)

print(f"Merged metadata: {len(df_merged)} entries.")
df_merged.head()

## 3. Multi-Channel Data Discovery

BBBC021 uses three channels (Actin, Tubulin, DAPI). We specify the filename columns for all three.

In [ ]:
from phenome.utils import make_dataframe_metadata_fn

file_df = pheno.find_files(
    str(DATA_DIR),
    metadata_fn=make_dataframe_metadata_fn(
        df_merged,
        filename_column=["Image_FileName_Actin", "Image_FileName_Tubulin", "Image_FileName_DAPI"],
    ),
)
file_df.head()

## 4. Feature Extraction

Extract embeddings and compute morphological properties.

In [ ]:
pheno.process_images(
    wrapper,
    batch_size=16,
    channel_mode="split",
    resize_size=1036,
    pad_size=1280,
    checkpoint_path=CHECKPOINT_EMBEDDINGS,
)

df_properties = pheno.compute_properties(
    property_preset="full_extended",
    checkpoint_path=CHECKPOINT_PROPERTIES,
    n_jobs=4,
)

## 5. Group Statistics

Calculate statistics grouped by compound and mechanism of action.

In [ ]:
df_grouped = pheno.property_stats_by_group(
    group_by=["Image_Metadata_Compound", "Mechanism_of_action"],
    print_properties=["intensity_mean_ch0", "intensity_std_ch0"],
)

## 6. Interactive Exploration

Explore the 3-channel phenospace interactively.

In [ ]:
pheno.create_interactive_explorer(
    hover_features=["Image_Metadata_Concentration", "Image_Metadata_Compound"],
)